# 📚 Books to Scrape — Web Scraping with Python

This notebook scrapes all books from [books.toscrape.com](https://books.toscrape.com) and saves the results to a CSV file.

**What we extract per book:**
- Title (full, not truncated)
- Price (as a float)
- Star rating (as an integer 1–5)
- Availability
- URL

**Libraries used:** `requests`, `beautifulsoup4`, `pandas`, `matplotlib`

## 1. Install & Import Libraries

In [ ]:
# Run this cell only if libraries are not installed
!pip install requests beautifulsoup4 pandas matplotlib

In [ ]:
import csv
import requests
import pandas as pd
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
from urllib.parse import urljoin

print("✅ All libraries imported successfully!")

## 2. Configuration

In [ ]:
BASE_URL    = "https://books.toscrape.com/"
OUTPUT_FILE = "books.csv"

# Maps rating words to integers
RATING_MAP = {
    "Zero":  0,
    "One":   1,
    "Two":   2,
    "Three": 3,
    "Four":  4,
    "Five":  5,
}

# Polite browser-like User-Agent header
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )
}

print("✅ Configuration set!")

## 3. Fetch the Page

In [ ]:
def fetch_page(url):
    """Fetch the HTML content of a page."""
    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
        response.raise_for_status()
        print(f"✅ Status: {response.status_code} | Size: {len(response.text):,} chars")
        return response.text
    except requests.RequestException as e:
        print(f"❌ Error: {e}")
        return None

html = fetch_page(BASE_URL)

## 4. Parse HTML & Find Book Cards

In [ ]:
soup  = BeautifulSoup(html, "html.parser")
cards = soup.find_all("article", class_="product_pod")

print(f"📦 Found {len(cards)} book cards on the homepage")
print(f"\nFirst card HTML preview:")
print(cards[0].prettify()[:500])

## 5. Extraction Functions

One small function per field — clean and easy to test individually.

In [ ]:
def extract_title(card):
    """Full title from the 'title' attribute (visible text is truncated)."""
    return card.find("h3").find("a")["title"]

def extract_price(card):
    """Price as a float, with £ sign stripped."""
    raw = card.find("p", class_="price_color").get_text(strip=True)
    return float(raw.replace("\u00a3", "").replace("£", ""))

def extract_rating(card):
    """Star rating as integer 0–5."""
    classes = card.find("p", class_="star-rating")["class"]
    return RATING_MAP.get(classes[1], 0)

def extract_availability(card):
    """Availability text (e.g. 'In stock')."""
    return card.find("p", class_="instock availability").get_text(strip=True)

def extract_url(card, base_url):
    """Full absolute URL to the book's detail page."""
    relative = card.find("h3").find("a")["href"]
    return urljoin(base_url, relative)

def extract_book(card, base_url):
    """Extract all fields from one book card and return as a dict."""
    return {
        "title":        extract_title(card),
        "price":        extract_price(card),
        "rating":       extract_rating(card),
        "availability": extract_availability(card),
        "url":          extract_url(card, base_url),
    }

# Quick test on the first card
print("🧪 Test on first card:")
print(extract_book(cards[0], BASE_URL))

## 6. Scrape All Books

In [ ]:
def stars(rating):
    return "\u2605" * rating + "\u2606" * (5 - rating)

books = []

for i, card in enumerate(cards, start=1):
    book = extract_book(card, BASE_URL)
    books.append(book)
    title = book["title"][:45].ljust(45)
    print(f"{i:>3}. {title}  £{book['price']:>6.2f}  {stars(book['rating'])}")

print(f"\n✅ Scraped {len(books)} books total")

## 7. Load into a DataFrame & Preview

In [ ]:
df = pd.DataFrame(books)
print(f"Shape: {df.shape}")
df.head(10)

## 8. Summary Statistics

In [ ]:
print("📊 Summary Statistics")
print("=" * 35)
print(f"Total books     : {len(df)}")
print(f"Average price   : £{df['price'].mean():.2f}")
print(f"Cheapest book   : £{df['price'].min():.2f}")
print(f"Most expensive  : £{df['price'].max():.2f}")
print(f"Average rating  : {df['rating'].mean():.2f} / 5")
print()
print("Rating distribution:")
print(df['rating'].value_counts().sort_index())

## 9. Visualisations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("Books to Scrape — Data Overview", fontsize=14, fontweight="bold")

# 1. Price distribution
axes[0].hist(df["price"], bins=15, color="steelblue", edgecolor="white")
axes[0].set_title("Price Distribution")
axes[0].set_xlabel("Price (£)")
axes[0].set_ylabel("Number of Books")

# 2. Rating distribution
rating_counts = df["rating"].value_counts().sort_index()
axes[1].bar(rating_counts.index, rating_counts.values, color="coral", edgecolor="white")
axes[1].set_title("Rating Distribution")
axes[1].set_xlabel("Star Rating")
axes[1].set_ylabel("Number of Books")
axes[1].set_xticks([1, 2, 3, 4, 5])

# 3. Average price per rating
avg_price = df.groupby("rating")["price"].mean()
axes[2].bar(avg_price.index, avg_price.values, color="mediumseagreen", edgecolor="white")
axes[2].set_title("Avg Price by Rating")
axes[2].set_xlabel("Star Rating")
axes[2].set_ylabel("Average Price (£)")
axes[2].set_xticks([1, 2, 3, 4, 5])

plt.tight_layout()
plt.savefig("books_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print("📈 Chart saved as books_analysis.png")

## 10. Save to CSV

In [ ]:
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")
print(f"💾 Saved {len(df)} books to '{OUTPUT_FILE}'")
print("\nFirst few rows of CSV:")
print(df[["title", "price", "rating"]].head())

## 11. (Colab only) Download the CSV

In [ ]:
# Only run this cell if you are on Google Colab
try:
    from google.colab import files
    files.download(OUTPUT_FILE)
    print("📥 Download started!")
except ImportError:
    print("ℹ️ Not running on Colab — find your CSV in the same folder as this notebook.")